# LP3 Practical 12 – K-Means and Hierarchical Clustering
**Aim:** Apply K-Means and Hierarchical Clustering to `sales_data_sample.csv` and determine the number of clusters using the Elbow Method.

**Dataset note:** The supplied CSV is a sales transaction dataset, not the Mall Customers dataset described in the manual. It does not contain customer gender, age, annual income or spending score. This notebook therefore aggregates the actual transaction rows into customer-level sales features.

## Cell 1 – Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage


## Cell 2 – Load and inspect dataset

In [ ]:
from pathlib import Path

def get_dataset_path(filename):
    candidates = [
        Path.cwd() / "Datasets" / filename,
        Path.cwd().parent / "Datasets" / filename
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Keep the Datasets folder beside the Notebooks folder."
    )

DATA_PATH = get_dataset_path("sales_data_sample.csv")
df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Dataset shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())


## Cell 3 – Clean required columns and create customer-level data

In [ ]:
required_columns = [
    "CUSTOMERNAME", "ORDERNUMBER", "QUANTITYORDERED",
    "SALES", "PRICEEACH"
]

sales = df[required_columns].copy()

sales["QUANTITYORDERED"] = pd.to_numeric(
    sales["QUANTITYORDERED"], errors="coerce"
)
sales["SALES"] = pd.to_numeric(sales["SALES"], errors="coerce")
sales["PRICEEACH"] = pd.to_numeric(sales["PRICEEACH"], errors="coerce")

sales = sales.dropna(subset=["CUSTOMERNAME", "ORDERNUMBER", "QUANTITYORDERED", "SALES"])

# One row per customer, created by aggregating their transactions.
customer_data = sales.groupby("CUSTOMERNAME").agg(
    TotalSales=("SALES", "sum"),
    TotalQuantity=("QUANTITYORDERED", "sum"),
    NumberOfOrders=("ORDERNUMBER", "nunique"),
    AveragePrice=("PRICEEACH", "mean")
).reset_index()

customer_data["AverageOrderValue"] = (
    customer_data["TotalSales"] / customer_data["NumberOfOrders"]
)

customer_data = customer_data.replace([np.inf, -np.inf], np.nan).dropna()

print("Number of customers:", len(customer_data))
display(customer_data.head())


## Cell 4 – Select features and scale them

In [ ]:
features = [
    "TotalSales", "TotalQuantity",
    "NumberOfOrders", "AveragePrice", "AverageOrderValue"
]

X = customer_data[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled feature shape:", X_scaled.shape)


## Cell 5 – Find the number of clusters using Elbow Method

In [ ]:
inertia_values = []
max_k = min(10, len(customer_data) - 1)

for k in range(1, max_k + 1):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, max_k + 1), inertia_values, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.xticks(range(1, max_k + 1))
plt.grid(True)
plt.show()


## Cell 6 – Apply K-Means clustering

In [ ]:
# Choose K by observing the elbow in the plot.
# Four clusters are used as a starting value; change it if your elbow suggests otherwise.
chosen_k = 4

kmeans_model = KMeans(
    n_clusters=chosen_k, random_state=42, n_init=10
)
customer_data["KMeans_Cluster"] = kmeans_model.fit_predict(X_scaled)

print(customer_data["KMeans_Cluster"].value_counts().sort_index())
display(customer_data.head())


## Cell 7 – Visualize K-Means clusters

In [ ]:
plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    customer_data["TotalSales"],
    customer_data["TotalQuantity"],
    c=customer_data["KMeans_Cluster"]
)
plt.xlabel("Total Sales")
plt.ylabel("Total Quantity")
plt.title("Customer Segments using K-Means")
plt.colorbar(scatter, label="Cluster")
plt.show()


## Cell 8 – Apply Hierarchical clustering

In [ ]:
hierarchical_model = AgglomerativeClustering(n_clusters=chosen_k)
customer_data["Hierarchical_Cluster"] = hierarchical_model.fit_predict(X_scaled)

print(customer_data["Hierarchical_Cluster"].value_counts().sort_index())
display(customer_data[[
    "CUSTOMERNAME", "KMeans_Cluster", "Hierarchical_Cluster"
]].head(10))


## Cell 9 – Draw a dendrogram

In [ ]:
# Dendrogram is displayed for the customer observations in this dataset.
linked = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(linked, truncate_mode="lastp", p=30)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Customer groups")
plt.ylabel("Distance")
plt.show()


## Cell 10 – Compare cluster quality

In [ ]:
if len(set(customer_data["KMeans_Cluster"])) > 1:
    print(
        "K-Means silhouette score:",
        silhouette_score(X_scaled, customer_data["KMeans_Cluster"])
    )

if len(set(customer_data["Hierarchical_Cluster"])) > 1:
    print(
        "Hierarchical silhouette score:",
        silhouette_score(X_scaled, customer_data["Hierarchical_Cluster"])
    )


## Sample output / observation
- The Elbow Method graph helps choose a suitable number of clusters.
- K-Means and Hierarchical Clustering assign a cluster label to each customer.
- Cluster labels are group identifiers; a larger label does not mean a better customer.
- The input file is transactional sales data, so the customer features used here are derived from its available columns.